In [1]:
import pandas as pd, numpy as np

df = pd.read_csv(r"D:\MBFM_project\master_features.csv", index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("\nNaN counts in HAR columns:")
cols = ["rv", "rv_lag1", "rv_5d", "rv_22d", "vix_level", "ret_t"]
print(df[cols].isna().sum())

print("\nFirst valid row index:")
print(df[cols].dropna().index[0])

print("\nRows 1495–1505 of rv_lag1:")
print(df["rv_lag1"].iloc[1495:1506])

print("\nRows 1495–1505 of rv:")
print(df["rv"].iloc[1495:1506])

Shape: (5199, 24)

NaN counts in HAR columns:
rv           0
rv_lag1      0
rv_5d        0
rv_22d       0
vix_level    0
ret_t        0
dtype: int64

First valid row index:
2005-04-04 00:00:00

Rows 1495–1505 of rv_lag1:
date
2011-06-02    0.046670
2011-06-03    0.045966
2011-06-06    0.046821
2011-06-07    0.044505
2011-06-08    0.044468
2011-06-09    0.042589
2011-06-10    0.043082
2011-06-13    0.043020
2011-06-14    0.044350
2011-06-15    0.044445
2011-06-16    0.043861
Name: rv_lag1, dtype: float64

Rows 1495–1505 of rv:
date
2011-06-02    0.045966
2011-06-03    0.046821
2011-06-06    0.044505
2011-06-07    0.044468
2011-06-08    0.042589
2011-06-09    0.043082
2011-06-10    0.043020
2011-06-13    0.044350
2011-06-14    0.044445
2011-06-15    0.043861
2011-06-16    0.045477
Name: rv, dtype: float64


In [2]:
print(df.columns.tolist())
print(df.iloc[1498:1502][["rv","rv_lag1","rv_5d","rv_22d"]].to_string())

['rv', 'rv_lag1', 'rv_5d', 'rv_22d', 'sq_return', 'vix_level', 'vix_chg', 'brent_ret', 'dxy_level', 'dxy_ret', 'india10y', 'india10y_chg', 'us10y', 'us10y_chg', 'rate_spread', 'callmoney', 'nifty_ret', 'em_ret', 'ret_t', 'is_mpc_day', 'is_fomc_day', 'day_of_week', 'month', 'week_of_year']
                  rv   rv_lag1     rv_5d    rv_22d
date                                              
2011-06-07  0.044468  0.044505  0.045768  0.055059
2011-06-08  0.042589  0.044468  0.045686  0.054014
2011-06-09  0.043082  0.042589  0.044870  0.052815
2011-06-10  0.043020  0.043082  0.044293  0.051654


In [2]:
import pandas as pd, numpy as np, statsmodels.api as sm

df = pd.read_csv(r"D:\MBFM_project\master_features.csv", index_col=0, parse_dates=True)
rv   = df["rv"].values
rv_d = df["rv_lag1"].values
rv_w = df["rv_5d"].values
rv_m = df["rv_22d"].values

TRAIN = 1500
t = TRAIN
start = 22

y = rv[start:t]
X = sm.add_constant(np.column_stack([rv_d[start:t], rv_w[start:t], rv_m[start:t]]))

print("y shape:", y.shape, "any nan:", np.isnan(y).any())
print("X shape:", X.shape, "any nan:", np.isnan(X).any())

res = sm.OLS(y, X).fit()
print("params:", res.params)

x_new = np.array([[1, rv_d[t], rv_w[t], rv_m[t]]])
pred  = float(res.predict(x_new))
print("first OOS prediction:", pred)

y shape: (1478,) any nan: False
X shape: (1478, 4) any nan: False
params: [ 8.17641913e-04  1.12390680e+00 -1.13142213e-01 -2.31013452e-02]


TypeError: only 0-dimensional arrays can be converted to Python scalars

In [ ]:
import numpy as np, pandas as pd, os

FCAST = r"D:\MBFM_project\results\forecasts"
DATA  = r"D:\MBFM_project\master_features.csv"

df     = pd.read_csv(DATA, index_col=0, parse_dates=True)
actual = df["rv"].values[1500:]

files = {
    "GARCH(1,1)" : "garch11.npy",
    "EGARCH"     : "egarch.npy",
    "GJR-GARCH"  : "gjr_garch.npy",
    "HAR-RV"     : "har_rv.npy",
    "HAR-RV-X"   : "har_rv_x.npy",
    "random_forest" : "random_forest.npy"
}

print(f"{'Model':<15} {'OOS_n':>6} {'RMSE':>10} {'MAE':>10} {'QLIKE':>10}")
print("-" * 55)
for name, fname in files.items():
    path = os.path.join(FCAST, fname)
    if not os.path.exists(path):
        print(f"{name:<15} {'MISSING':>6}")
        continue
    fc   = np.load(path)
    mask = ~np.isnan(fc) & (fc > 0)
    if mask.sum() == 0:
        print(f"{name:<15} {'n=0':>6}")
        continue
    a, f = actual[mask], fc[mask]
    rmse  = np.sqrt(np.mean((f - a)**2))
    mae   = np.mean(np.abs(f - a))
    qlike = np.mean(a/f - np.log(a/f) - 1)
    print(f"{name:<15} {mask.sum():>6} {rmse:>10.6f} {mae:>10.6f} {qlike:>10.6f}")

Model            OOS_n       RMSE        MAE      QLIKE
-------------------------------------------------------
GARCH(1,1)        3699   0.033610   0.025026   0.154288
EGARCH            3699   0.031356   0.023141   0.131604
GJR-GARCH         3699   0.035170   0.026186   0.158456
HAR-RV            3699   0.003470   0.001943   0.002243
HAR-RV-X          3699   0.003484   0.001982   0.002262
